## IA Skill Testing

_opensimula: v 0.6.1_

Fue gerenado usando este Prompt:

"\OpenSimula podrías crearme un diccionario con un proyecto que incluya un edificio de 50 m^2 y sistema HVAC perfecto, simularlo y mostrarme los valores horarios de la temperatura del espacio y la carga térmica sobre el mismo en una gráfica"

In [1]:
import opensimula.Simulation as Simulation

sim = Simulation()
pro = sim.new_project("Edificio 50m2")

project_dict = {
    "name": "Edificio 50m2",
    "time_step": 3600,
    "n_time_steps": 8760,
    "initial_time": "01/01/2001 00:00:00",
    "simulation_file_met": "met",
    "shadow_calculation": "INTERPOLATION",
    "components": [
        # ── Clima ──────────────────────────────────────────────────────────
        {
            "type": "File_met", "name": "met",
            "file_name": "../../mets/sevilla.met",
            "file_type": "MET"
        },
        # ── Horarios de ocupación (lunes-viernes 8-18h) ───────────────────
        {
            "type": "Day_schedule", "name": "dia_laborable",
            "time_steps": [8*3600, 10*3600],
            "values": [0, 1, 0],
            "interpolation": "STEP"
        },
        {
            "type": "Day_schedule", "name": "dia_festivo",
            "time_steps": [],
            "values": [0],
            "interpolation": "STEP"
        },
        {
            "type": "Week_schedule", "name": "semana_tipo",
            "days_schedules": [
                "dia_laborable", "dia_laborable", "dia_laborable",
                "dia_laborable", "dia_laborable", "dia_festivo", "dia_festivo"
            ]
        },
        {
            "type": "Year_schedule", "name": "horario_ocupacion",
            "periods": [],
            "weeks_schedules": ["semana_tipo"]
        },
        # ── Materiales ─────────────────────────────────────────────────────
        {
            "type": "Material", "name": "hormigon",
            "conductivity": 1.95, "density": 2240, "specific_heat": 900
        },
        {
            "type": "Material", "name": "ladrillo",
            "conductivity": 0.81, "density": 1600, "specific_heat": 840
        },
        {
            "type": "Material", "name": "aislante",
            "conductivity": 0.04, "density": 30, "specific_heat": 1000
        },
        {
            "type": "Material", "name": "yeso",
            "conductivity": 0.57, "density": 1100, "specific_heat": 1000
        },
        # ── Construcciones ─────────────────────────────────────────────────
        {
            "type": "Construction", "name": "const_muro_ext",
            "solar_alpha": [0.6, 0.6],
            "lw_epsilon": [0.9, 0.9],
            "materials": ["ladrillo", "aislante", "yeso"],
            "thicknesses": [0.15, 0.06, 0.015]
        },
        {
            "type": "Construction", "name": "const_cubierta",
            "solar_alpha": [0.6, 0.6],
            "lw_epsilon": [0.9, 0.9],
            "materials": ["hormigon", "aislante", "yeso"],
            "thicknesses": [0.20, 0.08, 0.015]
        },
        {
            "type": "Construction", "name": "const_solera",
            "solar_alpha": [0.6, 0.6],
            "lw_epsilon": [0.9, 0.9],
            "materials": ["hormigon", "aislante"],
            "thicknesses": [0.15, 0.05]
        },
        # ── Ventanas: doble acristalamiento + marco PVC ────────────────────
        {
            "type": "Glazing", "name": "doble_vidrio",
            "solar_tau": 0.731,
            "solar_rho": [0.133, 0.133],
            "lw_epsilon": [0.837, 0.837],
            "g": [0.776, 0.776],
            "U": 2.914                             # W/(m²·K) doble acristalamiento
        },
        {
            "type": "Frame", "name": "marco_pvc",
            "solar_alpha": [0.6, 0.6],
            "lw_epsilon": [0.9, 0.9],
            "thermal_resistance": 0.35             # m²·K/W marco PVC
        },
        {
            "type": "Opening_type", "name": "ventana_doble",
            "glazing": "doble_vidrio",
            "frame": "marco_pvc",
            "glazing_fraction": 0.8,
            "frame_fraction": 0.2
        },
        # ── Edificio ───────────────────────────────────────────────────────
        {
            "type": "Building", "name": "edificio",
            "azimuth": 0,
            "ref_point": [0, 0, 0],
            "initial_temperature": 20
        },
        # ── Space_type con ganancias internas y infiltración reales ────────
        {
            "type": "Space_type", "name": "tipo_espacio",
            "input_variables": ["f = horario_ocupacion.values"],
            "people_density": "0.1 * f",
            "people_sensible": 75,
            "people_latent": 55,
            "people_radiant_fraction": 0.5,
            "light_density": "10 * f",
            "light_radiant_fraction": 0.5,
            "other_gains_density": "8 * f",
            "other_gains_radiant_fraction": 0.3,
            "other_gains_latent_fraction": 0.0,
            "infiltration": "0.3"
        },
        # ── Espacio: planta rectangular 5 x 10 m, altura 3 m ──────────────
        {
            "type": "Space", "name": "zona",
            "building": "edificio",
            "spaces_type": "tipo_espacio",
            "floor_area": 50,
            "volume": 150,
            "furniture_weight": 10
        },
        # ── Superficies ────────────────────────────────────────────────────
        # altitude: 0=cerramiento vertical, 90=cubierta (normal hacia arriba), -90=suelo (normal hacia abajo)
        {
            "type": "Building_surface", "name": "fachada_sur",
            "shape": "RECTANGLE", "width": 10.0, "height": 3.0,
            "ref_point": [0, 0, 0],
            "azimuth": 0, "altitude": 0,      # cerramiento vertical
            "surface_type": "EXTERIOR",
            "construction": "const_muro_ext",
            "spaces": ["zona"]
        },
        {
            "type": "Building_surface", "name": "fachada_norte",
            "shape": "RECTANGLE", "width": 10.0, "height": 3.0,
            "ref_point": [10, 5, 0],
            "azimuth": 180, "altitude": 0,        # cerramiento vertical
            "surface_type": "EXTERIOR",
            "construction": "const_muro_ext",
            "spaces": ["zona"]
        },
        {
            "type": "Building_surface", "name": "fachada_este",
            "shape": "RECTANGLE", "width": 5.0, "height": 3.0,
            "ref_point": [10, 0, 0],
            "azimuth": 90, "altitude": 0,       # cerramiento vertical
            "surface_type": "EXTERIOR",
            "construction": "const_muro_ext",
            "spaces": ["zona"]
        },
        {
            "type": "Building_surface", "name": "fachada_oeste",
            "shape": "RECTANGLE", "width": 5.0, "height": 3.0,
            "ref_point": [0, 5, 0],
            "azimuth": -90, "altitude": 0,      # cerramiento vertical
            "surface_type": "EXTERIOR",
            "construction": "const_muro_ext",
            "spaces": ["zona"]
        },
        {
            "type": "Building_surface", "name": "cubierta",
            "shape": "RECTANGLE", "width": 10.0, "height": 5.0,
            "ref_point": [0, 0, 3],
            "azimuth": 0, "altitude": 90,       # cubierta horizontal (normal hacia arriba)
            "surface_type": "EXTERIOR",
            "construction": "const_cubierta",
            "spaces": ["zona"]
        },
        {
            "type": "Building_surface", "name": "solera",
            "shape": "RECTANGLE", "width": 10.0, "height": 5.0,
            "ref_point": [0, 5, 0],
            "azimuth": 0, "altitude": -90,      # suelo (normal hacia abajo)
            "surface_type": "UNDERGROUND",
            "construction": "const_solera",
            "ground_material": "hormigon",
            "spaces": ["zona"]
        },
        # ── Ventanas Este y Oeste (1.5 x 1.2 m, centradas) ────────────────
        # Fachada 5 m ancho x 3 m alto → esquina inferior izq: x=(5-1.5)/2=1.75, y=(3-1.2)/2=0.9
        {
            "type": "Opening", "name": "ventana_este",
            "surface": "fachada_este",
            "shape": "RECTANGLE", "width": 1.5, "height": 1.2,
            "ref_point": [1.75, 0.9],
            "opening_type": "ventana_doble",
            "setback": 0.1
        },
        {
            "type": "Opening", "name": "ventana_oeste",
            "surface": "fachada_oeste",
            "shape": "RECTANGLE", "width": 1.5, "height": 1.2,
            "ref_point": [1.75, 0.9],
            "opening_type": "ventana_doble",
            "setback": 0.1
        },
        # ── Sistema HVAC perfecto ──────────────────────────────────────────
        {
            "type": "HVAC_perfect_system", "name": "hvac",
            "spaces": "zona",
            "input_variables": ["f = horario_ocupacion.values"],
            "outdoor_air_flow": "0.05 * f",
            "heating_setpoint": "20",
            "cooling_setpoint": "25"
        }
    ]
}

pro.read_dict(project_dict)
pro.simulate()

Reading project data from dictonary
Reading completed.
Checking project: Edificio 50m2
Checking completed.
Calculating solar direct shadows ...
Simulating Edificio 50m2: ...


100%|██████████| 8760/8760 [00:09<00:00, 881.78step/s, n_iter=3] 


In [2]:
#pro.show_3D()

In [3]:
zona = pro.component("zona")
hvac = pro.component("hvac")

sim.plot(
    pro.dates(),
    [
        zona.variable("temperature"),
        hvac.variable("Q_sensible"),
    ],
    names=["Temperatura zona [°C]", "Carga térmica HVAC [W]"],
    axis=[1, 2],          # ejes Y independientes
    frequency=None,       # valores horarios
)

In [4]:
zona = pro.component("zona")

sim.plot(
    pro.dates(),
    [zona.variable("solar_direct_gains")],
    names=["Ganancias solares directas [W]"],
    frequency=None,
)